# SC203 DoSSR — LoRA 2 vs. LoRA 3 Controlled Colab Pilot & Bicubic Floor (`03_lora_pilot_colab.ipynb`)

**Team:** LEMMEPASS (Team 1, SC203 Scientific Methods — Class 25A02)  
**Target Branch:** `feat/baseline-runner`  
**Purpose:** End-to-end Google Colab L4 GPU notebook for:
1. Environment / GPU / Drive verification + **Bicubic $\times 4$ whole-image floor** (`RealSRVal_crop128`) for Assignment 5 Table 1.
2. **Scene-level Real-CE $\times 4$ `train` split & patch extraction** (`scripts/split_realce_by_scene.py`, zero scene leakage, `test` held out untouched).
3. **LoRA 2 pilot training** (text-weighted reconstruction loss, $\lambda_{\text{cons}} = 0$).
4. **LoRA 3 pilot training** (same init, data, empty prompt `""`, $K=2$ noise draws, `seed=231`, 400 steps + cross-noise consistency).
5. **Fixed-validation 10-seed (`42..51`), 5-step inference** (`per_image_seed: true`) + full-reference metrics + RapidOCR evaluation.
6. **Decision gate & mentor-update table** (delta Error Rate, CER, Seed Disagreement, LPIPS, Peak VRAM GiB, and mean/p95 `s/step`).

## CELL 01 — Mount Google Drive, Pull Branch & Check L4 GPU / Dependencies

In [ ]:
# === CELL 01: Mount Drive, clone/pull SC203, verify L4 GPU & dependencies ===
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/SC203')
DRIVE_BASE = Path('/content/drive/MyDrive/lemmepass')
DATA_DIR = DRIVE_BASE / 'data'
OUT_BASE = DRIVE_BASE / 'outputs'
CKPT_PATH = DRIVE_BASE / 'weights' / 'dossr_default.ckpt'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', 'feat/baseline-runner', 'https://github.com/lemon4life/SC203.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GiB')
print('Checkpoint exists:', CKPT_PATH.is_file(), '| path:', CKPT_PATH)

## CELL 02 — CPU Sanity Verification (`src/lora_text_loss.py` & `scripts/run_lora_trial.py`)

In [ ]:
# === CELL 02: Run offline CPU unit tests & training pipeline dry run ===
import subprocess
import sys

def run_streamed(cmd):
    print('RUN:', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}')

run_streamed([sys.executable, '-u', 'src/lora_text_loss.py', '--run-cpu-tests'])
run_streamed([sys.executable, '-u', 'scripts/run_lora_trial.py', '--dry-run-cpu'])

## CELL 03 — Fill Assignment 5 Table 1: Bicubic $\times 4$ Whole-Image Floor (`RealSRVal_crop128`)

In [ ]:
# === CELL 03: Compute Bicubic x4 whole-image PSNR_RGB, SSIM_RGB, PSNR_Y, SSIM_Y, LPIPS on RealSRVal_crop128 ===
REALSR_ROOT = DATA_DIR / 'StableSR_testsets' / 'StableSR_testsets' / 'RealSRVal_crop128'
BICUBIC_SUMMARY_CSV = REPO_DIR / 'results' / 'metrics_realsr_bicubic_x4_summary.csv'

run_streamed([
    sys.executable, '-u', 'scripts/compute_metrics.py',
    '--bicubic-lr-dir', str(REALSR_ROOT / 'test_LR'),
    '--gt-dir', str(REALSR_ROOT / 'test_HR'),
    '--expected-size', '512',
    '--device', 'cuda',
    '--summary-csv', str(BICUBIC_SUMMARY_CSV),
])

## CELL 04 — Scene-Level Real-CE $\times 4$ `train` Split & Aligned Patch Extraction (Zero Leakage)

In [ ]:
# === CELL 04: Partition Real-CE train split by scene (holding out test) & extract aligned 128->512 patches ===
REALCE_DIR = DATA_DIR / 'Real-CE'
REALCE_PATCH_DIR = DATA_DIR / 'RealCE_patches'

run_streamed([
    sys.executable, '-u', 'scripts/split_realce_by_scene.py',
    '--data-dir', str(REALCE_DIR),
    '--split', 'train',
    '--scale', 'x4',
    '--val-ratio', '0.20',
    '--seed', '42',
    '--output-manifest', 'data/realce_splits_manifest.json',
    '--output-train-csv', 'data/realce_train_manifest.csv',
    '--output-val-boxes', 'data/realce_val_boxes.csv',
    '--extract-patches',
    '--patch-dir', str(REALCE_PATCH_DIR),
    '--hr-patch-size', '512',
    '--lr-patch-size', '128',
    '--stride', '256',
])

## CELL 05 — Train Controlled Pilot: LoRA 2 (Text-Weighted L1) vs. LoRA 3 (+ Cross-Noise Consistency)

In [ ]:
# === CELL 05: Train LoRA 2 and LoRA 3 under identical init, data, empty prompt "", K=2 noise draws, seed=231, 400 steps ===
LORA2_OUT = OUT_BASE / 'lora2-pilot'
LORA3_OUT = OUT_BASE / 'lora3-pilot'

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora2_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(LORA2_OUT),
])

run_streamed([
    sys.executable, '-u', 'scripts/run_lora_trial.py',
    '--config', 'configs/lora3_pilot.yaml',
    '--train-manifest', 'data/realce_train_manifest.csv',
    '--output-dir', str(LORA3_OUT),
])

## CELL 06 — Fixed-Validation 10-Seed (`42..51`), 5-Step Inference & Quantitative Evaluation

In [ ]:
# === CELL 06: 10-seed 5-step validation inference + LPIPS/PSNR/SSIM + OCR comparison ===
VAL_LORA2_OUT = OUT_BASE / 'pilot-lora2-realce-val-10seed'
VAL_LORA3_OUT = OUT_BASE / 'pilot-lora3-realce-val-10seed'

run_streamed([sys.executable, '-u', 'scripts/run_seed_study.py', '--config', 'configs/lora2_pilot.yaml'])
run_streamed([sys.executable, '-u', 'scripts/run_seed_study.py', '--config', 'configs/lora3_pilot.yaml'])

for label, study_out in [('lora2', VAL_LORA2_OUT), ('lora3', VAL_LORA3_OUT)]:
    run_streamed([
        sys.executable, '-u', 'scripts/compute_metrics.py',
        '--output-root', str(study_out),
        '--gt-dir', str(DATA_DIR / 'RealCE_patches' / 'val' / 'HR'),
        '--expected-size', '512',
        '--test-y-channel',
        '--summary-csv', str(REPO_DIR / 'results' / f'metrics_pilot_{label}_val_summary.csv'),
        '--per-image-csv', str(REPO_DIR / 'results' / f'metrics_pilot_{label}_val_per_seed.csv'),
    ])

run_streamed([
    sys.executable, '-u', 'scripts/text_eval.py',
    '--study-root', str(VAL_LORA3_OUT),
    '--compare-study-root', str(VAL_LORA2_OUT),
    '--boxes-csv', 'data/realce_val_boxes.csv',
    '--output-dir', 'results/pilot_lora2_vs_lora3',
])

## CELL 07 — Pilot Decision Gate & Mentor Update Summary (VRAM & `s/step`)

In [ ]:
# === CELL 07: Print Peak VRAM, step time, LPIPS, and OCR comparison table for Dr. Thuan update ===
import csv
import json

for name, out_dir in [('LoRA 2', LORA2_OUT), ('LoRA 3', LORA3_OUT)]:
    m_path = out_dir / 'lora_run_manifest.json'
    if m_path.is_file():
        m = json.loads(m_path.read_text(encoding='utf-8'))
        print(
            f"{name:8s} | params={m['trainable_lora_params']:,} | "
            f"mean={m['mean_step_time_sec']:.2f}s/step (p95={m['p95_step_time_sec']:.2f}s) | "
            f"peak_alloc={m['peak_vram_allocated_gib']:.2f} GiB | "
            f"peak_res={m['peak_vram_reserved_gib']:.2f} GiB"
        )

for label in ('lora2', 'lora3'):
    summary_csv = REPO_DIR / 'results' / f'metrics_pilot_{label}_val_summary.csv'
    if summary_csv.is_file():
        rows = list(csv.DictReader(summary_csv.open(encoding='utf-8')))
        overall = next((r for r in rows if r['image_stem'] == 'ALL_IMAGES_MEAN'), None)
        if overall:
            print(
                f"{label.upper():8s} VAL | PSNR_Y={float(overall['psnr_y_mean']):.4f} | "
                f"SSIM_Y={float(overall['ssim_y_mean']):.4f} | LPIPS={float(overall['lpips_mean']):.4f}"
            )